In [1]:
import numpy as np
import pandas as pd
from unicodedata import normalize
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset,DataLoader
import torch

In [2]:
# Asegurarse de correr el script de limpieza para poder ejecutar el script a continuación
%run limpieza_data.ipynb

Todos los archivos han sido cargados
dataframe expextativas limpio
dataframe tasa_politica limpio
dataframe indice_precios limpio
dataframe tasa_ibr limpio
dataframe tasa_mercado limpio
Filtro temporal aplicado
Valores del mes seleccioandos
DATAFRAME CONSOLIDADO
Columna fecha eliminada de df_modelo_sin_fecha


## Realizar normalización de los datos

In [3]:
# Configuración de parametros para la clase BanrepDataset
n = len(df_modelo_sin_fecha)
train_obs = int(n * 0.70)
val_obs = int(n * 0.85)
test_obs = n - train_obs - val_obs

scaler = StandardScaler()
scaler.fit(df_modelo_sin_fecha[:train_obs])
df_modelo_sin_fecha = scaler.transform(df_modelo_sin_fecha)
ipc_mean = scaler.mean_[2]
ipc_std = scaler.scale_[2]

## Construir la clase que hereda atributos de Dataset

In [4]:
pasos_dato = 1
contexto = 12
retraso = pasos_dato * (contexto + 1 - 1)
batch_size = 32

class BanrepDataset(Dataset):
    def __init__(self, data, sequence_length, target_col, indice_inicio, indice_final, sampling_rate):
        self.data = data
        self.sequence_length = sequence_length
        self.target = target_col    
        self.indices = np.arange(indice_inicio, indice_final)
        self.sampling_rate   = sampling_rate
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, index):
        inicio = self.indices[index]
        pasos = np.arange(inicio, inicio + self.sequence_length * self.sampling_rate, self.sampling_rate)

        x = self.data[pasos]
        y = self.data[inicio + retraso, self.target]
        return torch.tensor(x, dtype = torch.float32), torch.tensor(y, dtype = torch.float32)

## Construir cada uno de los datasets con las caracteristicas necesarias para el modelo

In [5]:
train_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = 0,
    indice_final = train_obs,
    sampling_rate = pasos_dato
)

val_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = train_obs - retraso,
    indice_final = val_obs - retraso,
    sampling_rate = pasos_dato
)

test_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = val_obs - retraso,
    indice_final = n - retraso,
    sampling_rate = pasos_dato
)

## Crear los dataloaders para envíar la información al modelo

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = False)
val_loader = DataLoader(val_dataset,   batch_size = batch_size, shuffle = False)
test_loader = DataLoader(test_dataset,  batch_size = batch_size, shuffle = False)

In [7]:
# Chequeo de que los dataloaders tengan las dimensiones correctas
for entrada, vble_predecir in train_dataloader:
    print('Dimensiones entradas: ', entrada.shape)
    print('Dimensión variable objetivo: ', vble_predecir.shape)
    break

Dimensiones entradas:  torch.Size([32, 12, 5])
Dimensión variable objetivo:  torch.Size([32])


## Modelo de presistencia

In [ ]:
def modelo_persistencia(contenedor):
    total_error_abs = 0.0
    muestras_vistas = 0
    with torch.no_grad():
        for muestra, objetivo in contenedor:
            predicciones = muestra[:, -1, 1] * ipc_std + ipc_mean 
            objetivo_desnorm = objetivo * ipc_std + ipc_mean
            # Tome el ultimo valor de la TPM de todos los batch en el ultimo paso temporal
            total_error_abs += torch.sum(torch.abs(predicciones - objetivo_desnorm)).item()
            muestras_vistas += muestra.shape[0]
    
    return total_error_abs / muestras_vistas

print(f'MAE en validación, modelo de persistencia: {modelo_persistencia(val_loader):.2f}')
print(f'MAE en test, modelo de persistencia: {modelo_persistencia(test_loader):.2f}')
# NOTA: el resultado esta en puntos, por lo que el MAE se puede interpretar de dicha manera

MAE en validación, modelo de persistencia: 26.44
MAE en test, modelo de persistencia: 23.48
